In [ ]:
# Initialize Earth Engine
import ee
try:
    ee.Initialize(project='river-468515')
    print("Earth Engine initialized successfully!")
except:
    ee.Authenticate()
    ee.Initialize(project='river-468515')
    print("Earth Engine authenticated and initialized!")

Earth Engine authenticated and initialized!


In [ ]:
# ======================================================================
# YEARLY WATER MASK COMPOSITE EXPORT (1988–2025)
# Same filtering + same sources as your quarterly workflow
#
# 1988–2014  -> Landsat 5 / 7 / 8
# 2015–2025  -> Sentinel-2 primary + Sentinel-1 fallback
#
# Output:
# one yearly composite TIFF per year
# ======================================================================

import os
import ee
import geemap
import time
import numpy as np

# ======================================================================
# CONFIGURATION
# ======================================================================

study_area = ee.Geometry.Polygon([
    [88.75677098777692, 24.00347653384235],
    [90.59972753074567, 23.152644569433964],
    [90.52556981590192, 23.54602121781216],
    [88.74480244337559, 24.334092333430064],
    [88.75677098777692, 24.00347653384235]
])

START_YEAR = 1988
END_YEAR = 2025

out_dir = "./Yearly_WaterMasks_1988_2025"
os.makedirs(out_dir, exist_ok=True)

WATER_THRESHOLD = 0
CLOUD_COVER_MAX = 50
EXPORT_SCALE = 60

MAX_S2_IMAGES = 25
MAX_S1_IMAGES = 20

# ======================================================================
# HELPER FUNCTIONS
# ======================================================================

def mask_landsat_clouds(image):
    qa = image.select("QA_PIXEL")

    cloud_shadow = 1 << 3
    cloud = 1 << 4

    mask = qa.bitwiseAnd(cloud_shadow).eq(0).And(
        qa.bitwiseAnd(cloud).eq(0)
    )

    optical = image.select(
        ["Green", "SWIR1"]
    ).multiply(0.0000275).add(-0.2)

    return image.select("QA_PIXEL").addBands(
        optical,
        overwrite=True
    ).updateMask(mask)


def mask_sentinel2_clouds(image):
    qa = image.select("QA60")

    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11

    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
        qa.bitwiseAnd(cirrus_bit_mask).eq(0)
    )

    return image.updateMask(mask).divide(10000)


def add_sentinel1_ratio(image):
    vv = image.select("VV")
    vh = image.select("VH")
    ratio = vv.divide(vh).rename("VV_VH_ratio")
    return image.addBands(ratio)


# ======================================================================
# LANDSAT YEARLY (1988–2014)
# ======================================================================

def export_landsat_yearly(year):
    try:
        start_date = ee.Date.fromYMD(year, 1, 1)
        end_date = ee.Date.fromYMD(year, 12, 31).advance(1, "day")

        collections = []

        # Landsat 8
        if end_date.millis().getInfo() > ee.Date("2013-03-18").millis().getInfo():
            l8 = (
                ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
                .filterDate(start_date, end_date)
                .filterBounds(study_area)
                .filter(ee.Filter.lt("CLOUD_COVER", CLOUD_COVER_MAX))
                .select(
                    ["SR_B3", "SR_B6", "QA_PIXEL"],
                    ["Green", "SWIR1", "QA_PIXEL"]
                )
            )
            collections.append(l8)

        # Landsat 7
        l7 = (
            ee.ImageCollection("LANDSAT/LE07/C02/T1_L2")
            .filterDate(start_date, end_date)
            .filterBounds(study_area)
            .filter(ee.Filter.lt("CLOUD_COVER", CLOUD_COVER_MAX))
            .select(
                ["SR_B2", "SR_B5", "QA_PIXEL"],
                ["Green", "SWIR1", "QA_PIXEL"]
            )
        )
        collections.append(l7)

        # Landsat 5
        if start_date.millis().getInfo() < ee.Date("2013-06-05").millis().getInfo():
            l5 = (
                ee.ImageCollection("LANDSAT/LT05/C02/T1_L2")
                .filterDate(start_date, end_date)
                .filterBounds(study_area)
                .filter(ee.Filter.lt("CLOUD_COVER", CLOUD_COVER_MAX))
                .select(
                    ["SR_B2", "SR_B5", "QA_PIXEL"],
                    ["Green", "SWIR1", "QA_PIXEL"]
                )
            )
            collections.append(l5)

        if not collections:
            return "no_collections", 0, 0

        merged = collections[0]
        for col in collections[1:]:
            merged = merged.merge(col)

        masked = merged.map(mask_landsat_clouds)

        count = masked.size().getInfo()

        if count == 0:
            return "no_images", 0, 0

        composite = masked.median().clip(study_area)

        mndwi = composite.normalizedDifference(["Green", "SWIR1"])
        water_mask = mndwi.gt(WATER_THRESHOLD).rename("water").toByte()

        pixel_area = water_mask.multiply(ee.Image.pixelArea())

        water_area = pixel_area.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=study_area,
            scale=30,
            maxPixels=1e10,
            bestEffort=True
        )

        area_km2 = ee.Number(
            water_area.get("water", 0)
        ).divide(1e6).getInfo()

        if area_km2 == 0:
            return "no_water", count, 0

        out_tif = os.path.join(
            out_dir,
            f"water_mask_{year}.tif"
        )

        geemap.ee_export_image(
            water_mask.selfMask(),
            filename=out_tif,
            scale=EXPORT_SCALE,
            region=study_area,
            file_per_band=False,
            crs="EPSG:4326"
        )

        return "success", count, area_km2

    except Exception as e:
        return f"error: {str(e)[:30]}", 0, 0


# ======================================================================
# SENTINEL YEARLY (2015–2025)
# ======================================================================

def export_sentinel_yearly(year):
    try:
        start_date = ee.Date.fromYMD(year, 1, 1)
        end_date = ee.Date.fromYMD(year, 12, 31).advance(1, "day")

        # Sentinel-2
        s2 = (
            ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
            .filterDate(start_date, end_date)
            .filterBounds(study_area)
            .filter(
                ee.Filter.lt(
                    "CLOUDY_PIXEL_PERCENTAGE",
                    CLOUD_COVER_MAX
                )
            )
            .sort("CLOUDY_PIXEL_PERCENTAGE")
            .limit(MAX_S2_IMAGES)
            .map(mask_sentinel2_clouds)
            .select(["B3", "B11"], ["Green", "SWIR1"])
        )

        s2_count = s2.size().getInfo()

        # Sentinel-1 fallback
        s1 = (
            ee.ImageCollection("COPERNICUS/S1_GRD")
            .filterDate(start_date, end_date)
            .filterBounds(study_area)
            .filter(ee.Filter.eq("instrumentMode", "IW"))
            .filter(
                ee.Filter.listContains(
                    "transmitterReceiverPolarisation",
                    "VV"
                )
            )
            .filter(
                ee.Filter.listContains(
                    "transmitterReceiverPolarisation",
                    "VH"
                )
            )
            .filter(
                ee.Filter.eq(
                    "orbitProperties_pass",
                    "DESCENDING"
                )
            )
            .sort("system:time_start", False)
            .limit(MAX_S1_IMAGES)
            .select(["VV", "VH"])
            .map(add_sentinel1_ratio)
        )

        s1_count = s1.size().getInfo()

        if s2_count == 0 and s1_count == 0:
            return "no_images", 0, 0

        # Primary method
        if s2_count > 0:
            composite = s2.median().clip(study_area)
            mndwi = composite.normalizedDifference(["Green", "SWIR1"])
            water_mask = mndwi.gt(WATER_THRESHOLD).rename("water").toByte()
            method = "S2_MNDWI"

        else:
            composite = s1.median().clip(study_area)
            vv = composite.select("VV")
            water_mask = vv.lt(-16).rename("water").toByte()
            method = "S1_VV"

        pixel_area = water_mask.multiply(ee.Image.pixelArea())

        water_area = pixel_area.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=study_area,
            scale=30,
            maxPixels=1e10,
            bestEffort=True
        )

        area_km2 = ee.Number(
            water_area.get("water", 0)
        ).divide(1e6).getInfo()

        if area_km2 == 0:
            return "no_water", s2_count + s1_count, 0

        out_tif = os.path.join(
            out_dir,
            f"water_mask_{year}.tif"
        )

        geemap.ee_export_image(
            water_mask.selfMask(),
            filename=out_tif,
            scale=EXPORT_SCALE,
            region=study_area,
            file_per_band=False,
            crs="EPSG:4326"
        )

        return f"success ({method})", s2_count + s1_count, area_km2

    except Exception as e:
        return f"error: {str(e)[:30]}", 0, 0


# ======================================================================
# MAIN
# ======================================================================

print("=" * 70)
print(f"YEARLY WATER MASK EXPORT ({START_YEAR}-{END_YEAR})")
print("1988–2014 : Landsat")
print("2015–2025 : Sentinel")
print(f"Cloud Cover Max : {CLOUD_COVER_MAX}%")
print(f"Export Scale    : {EXPORT_SCALE}m")
print("=" * 70)

summary = {
    "success": 0,
    "failed": 0,
    "areas": []
}

for year in range(START_YEAR, END_YEAR + 1):

    out_tif = os.path.join(
        out_dir,
        f"water_mask_{year}.tif"
    )

    if os.path.exists(out_tif):
        print(f"{year} ⏭️ Exists")
        summary["success"] += 1
        continue

    print(f"{year} processing...", end=" ", flush=True)

    if year <= 2014:
        status, count, area = export_landsat_yearly(year)
    else:
        status, count, area = export_sentinel_yearly(year)

    if "success" in status:
        summary["success"] += 1
        summary["areas"].append(area)
        print(f"✅ {status} | {area:.2f} km² | {count} images")
    else:
        summary["failed"] += 1
        print(f"❌ {status}")

    time.sleep(2)

print("\n" + "=" * 70)
print("COMPLETE")
print("=" * 70)
print(f"Successful : {summary['success']}")
print(f"Failed     : {summary['failed']}")

if summary["areas"]:
    print(f"Average Water Area : {np.mean(summary['areas']):.2f} km²")
    print(f"Min Area           : {np.min(summary['areas']):.2f} km²")
    print(f"Max Area           : {np.max(summary['areas']):.2f} km²")

print(f"\nOutput Folder: {out_dir}")
print("=" * 70)

YEARLY WATER MASK EXPORT (1988-2025)
1988–2014 : Landsat
2015–2025 : Sentinel
Cloud Cover Max : 50%
Export Scale    : 60m
1988 processing... Generating URL ...
Please wait ...
Data downloaded to /content/Yearly_WaterMasks_1988_2025/water_mask_1988.tif
✅ success | 928.41 km² | 31 images
1989 processing... Generating URL ...
Please wait ...
Data downloaded to /content/Yearly_WaterMasks_1988_2025/water_mask_1989.tif
✅ success | 813.45 km² | 44 images
1990 processing... Generating URL ...
Please wait ...
Data downloaded to /content/Yearly_WaterMasks_1988_2025/water_mask_1990.tif
✅ success | 937.22 km² | 28 images
1991 processing... Generating URL ...
Please wait ...
Data downloaded to /content/Yearly_WaterMasks_1988_2025/water_mask_1991.tif
✅ success | 973.42 km² | 39 images
1992 processing... Generating URL ...
Please wait ...
Data downloaded to /content/Yearly_WaterMasks_1988_2025/water_mask_1992.tif
✅ success | 724.49 km² | 45 images
1993 processing... Generating URL ...
Please wait ...

In [ ]:
# ============================================================
# ZIP ALL EXPORTED WATER MASK FILES
# ============================================================

import os
import zipfile

# Change this if needed
source_folder = "./Yearly_WaterMasks_1988_2025"

# Output zip file name
zip_filename = "Yearly_WaterMasks_1988_2025.zip"


def zip_folder(folder_path, zip_name):
    with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)

                # Keep relative path inside zip
                arcname = os.path.relpath(file_path, folder_path)

                zipf.write(file_path, arcname)

    print(f"✅ ZIP created successfully: {zip_name}")
    print(f"📦 Saved at: {os.path.abspath(zip_name)}")


# Run
zip_folder(source_folder, zip_filename)

✅ ZIP created successfully: Yearly_WaterMasks_1988_2025.zip
📦 Saved at: /content/Yearly_WaterMasks_1988_2025.zip
